# Final Project: Data Pipeline
This project is a pipeline that collects NFL team statistics from NFL.com and saves them as dataframes for analysis. A Python script uses Selenium to load each team stats page for the current regular season, reads the tables with pandas, standardizes team names, and merges the categories into one table each for offense, defense, and special teams, with one row per team. We chose to do this because NFL statistics are typically spread across many separate pages and formatted for reading rather than analysis, and collecting them by hand is impractical. The pipeline puts them in one place, ready for analysis and predictions or for anyone who wants accessible team statistics.
***

***
The cell below will import all necessary packages for reading data into pandas dataframes. Selenium will allow us to scrape data from https://www.nfl.com/stats/team-stats, while other packages aid in that process.
***

In [1]:
from io import StringIO
from pathlib import Path
import re
import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

***
The next cell initializes what is needed to navigate through https://www.nfl.com/stats/team-stats, offering the configurables SEASON and SEASON_TYPE. SEASON can be set to any season that either been completed or is currently being played. SEASON_TYPE has the options reg, pre, and post for regular season, preseason, and postseason stats, respectively.
***

In [2]:
# Configuration
SEASON = 2026
SEASON_TYPE = "reg"

WEEK = "all"
BASE_URL = "https://www.nfl.com/stats/team-stats"

OUTPUT_DIR = Path("nfl_data")
OUTPUT_DIR.mkdir(exist_ok=True)

# Statistics to scrape for each group
CATEGORIES = {
    "offense": [
        "passing",
        "rushing",
        "receiving",
        "scoring",
        "downs",
    ],
    "defense": [
        "passing",
        "rushing",
        "receiving",
        "scoring",
        "tackles",
        "downs",
        "fumbles",
        "interceptions",
    ],
    "special-teams": [
        "field-goals",
        "scoring",
        "kickoffs",
        "kickoff-returns",
        "punting",
        "punt-returns",
    ],
}

OUTPUT_FILES = {
    "offense": "all_offensive_stats.pkl",
    "defense": "all_defensive_stats.pkl",
    "special-teams": "all_special_stats.pkl",
}

TEAM_NAMES = {
    "49ers": "San Francisco 49ers",
    "Cardinals": "Arizona Cardinals",
    "Falcons": "Atlanta Falcons",
    "Ravens": "Baltimore Ravens",
    "Bills": "Buffalo Bills",
    "Panthers": "Carolina Panthers",
    "Bears": "Chicago Bears",
    "Bengals": "Cincinnati Bengals",
    "Browns": "Cleveland Browns",
    "Cowboys": "Dallas Cowboys",
    "Broncos": "Denver Broncos",
    "Lions": "Detroit Lions",
    "Packers": "Green Bay Packers",
    "Texans": "Houston Texans",
    "Colts": "Indianapolis Colts",
    "Jaguars": "Jacksonville Jaguars",
    "Chiefs": "Kansas City Chiefs",
    "Raiders": "Las Vegas Raiders",
    "Chargers": "Los Angeles Chargers",
    "Rams": "Los Angeles Rams",
    "Dolphins": "Miami Dolphins",
    "Vikings": "Minnesota Vikings",
    "Patriots": "New England Patriots",
    "Saints": "New Orleans Saints",
    "Giants": "New York Giants",
    "Jets": "New York Jets",
    "Eagles": "Philadelphia Eagles",
    "Steelers": "Pittsburgh Steelers",
    "Seahawks": "Seattle Seahawks",
    "Buccaneers": "Tampa Bay Buccaneers",
    "Titans": "Tennessee Titans",
    "Commanders": "Washington Commanders",
}

***
The following cell defines the functions used for scraping. normalize_team normalizes the scraped team names to the strings set in TEAM_NAMES for consistency across dataframes. scrape_category scrapes https://www.nfl.com/stats/team-stats one page/category at a time, returning the dataframe that is scraped. merge_categories merges each of the dataframe categories for offensive, defensive, and special teams stats, storing those merged dataframes as all_offensive_stats.pkl, all_defensive_stats.pkl, and all_special_stats.pkl, respectively.
***

In [3]:
def normalize_team(value):
    value = re.sub(r"\s+", " ", str(value)).strip()
    value = re.sub(r"^Image\s*", "", value, flags=re.I)

    if value in TEAM_NAMES:
        return TEAM_NAMES[value]

    for short_name, full_name in TEAM_NAMES.items():
        if value.lower().endswith(short_name.lower()):
            return full_name

    return value


def scrape_category(driver, group, category):
    url = (
        f"{BASE_URL}/{group}/{category}/"
        f"{SEASON}/{SEASON_TYPE}/{WEEK}"
    )

    print(f"\nOpening {group} - {category}: {url}")
    driver.get(url)

    WebDriverWait(driver, 30).until(
        lambda d: len(d.find_elements(By.TAG_NAME, "table")) > 0
    )

    html = driver.page_source
    tables = pd.read_html(StringIO(html))

    if not tables:
        raise ValueError(f"No tables found for {group}/{category}")

    df = None

    for candidate in tables:
        candidate.columns = [
            " ".join(map(str, col)).strip()
            if isinstance(col, tuple)
            else str(col).strip()
            for col in candidate.columns
        ]

        if any(
            col.lower() == "team"
            or col.lower().endswith(" team")
            for col in candidate.columns
        ):
            df = candidate.copy()
            break

    if df is None:
        raise ValueError(
            f"No team statistics table found for {group}/{category}. "
            f"Page title: {driver.title}"
        )

    team_column = next(
        col for col in df.columns
        if col.lower() == "team"
        or col.lower().endswith(" team")
    )

    df = df.rename(columns={team_column: "Team"})
    df["Team"] = df["Team"].apply(normalize_team)

    # Prefix statistic columns to avoid naming conflicts when merging.
    df = df.rename(columns={
        col: f"{category}_{col}"
        for col in df.columns
        if col != "Team"
    })

    print(
        f"{group}/{category}: "
        f"{df.shape[0]} rows, {df.shape[1]} columns"
    )

    return df


def merge_categories(driver, group, categories):
    dataframes = {}

    # Scrape each category into its own DataFrame.
    for category in categories:
        dataframes[category] = scrape_category(
            driver, group, category
        )
        time.sleep(2)

    # Start with the first DataFrame and merge the rest.
    merged_df = dataframes[categories[0]]

    for category in categories[1:]:
        merged_df = merged_df.merge(
            dataframes[category],
            on="Team",
            how="outer",
            validate="one_to_one",
        )

    merged_df = (
        merged_df
        .sort_values("Team")
        .reset_index(drop=True)
    )

    # Save the merged DataFrame as a pickle.
    output_file = OUTPUT_DIR / OUTPUT_FILES[group]
    merged_df.to_pickle(output_file)

    print(f"\nCompleted {group}")
    print("Shape:", merged_df.shape)
    print("Unique teams:", merged_df["Team"].nunique())
    print("Saved to:", output_file.resolve())

    return merged_df


***
This cell calls main(), where the code actually executes, calling all of the previously defined functions, to scrape and store the data.
***

In [4]:
def main():
    options = webdriver.FirefoxOptions()
    service = Service("/snap/bin/geckodriver")
    driver = webdriver.Firefox(
        service=service,
        options=options,
    )

    all_dataframes = {}

    try:
        for group, categories in CATEGORIES.items():
            all_dataframes[group] = merge_categories(
                driver, group, categories
            )
    finally:
        driver.quit()

    print("\nAll datasets created successfully.")

    for group, df in all_dataframes.items():
        print(f"\n{group.upper()}")
        print("Shape:", df.shape)
        print(df.head())


if __name__ == "__main__":
    main()


Opening offense - passing: https://www.nfl.com/stats/team-stats/offense/passing/2026/reg/all
offense/passing: 32 rows, 16 columns

Opening offense - rushing: https://www.nfl.com/stats/team-stats/offense/rushing/2026/reg/all
offense/rushing: 32 rows, 11 columns

Opening offense - receiving: https://www.nfl.com/stats/team-stats/offense/receiving/2026/reg/all
offense/receiving: 32 rows, 11 columns

Opening offense - scoring: https://www.nfl.com/stats/team-stats/offense/scoring/2026/reg/all
offense/scoring: 32 rows, 5 columns

Opening offense - downs: https://www.nfl.com/stats/team-stats/offense/downs/2026/reg/all
offense/downs: 32 rows, 10 columns

Completed offense
Shape: (32, 49)
Unique teams: 32
Saved to: /home/matthewlieb/DS364/DS364_Final_Project/nfl_data/all_offensive_stats.pkl

Opening defense - passing: https://www.nfl.com/stats/team-stats/defense/passing/2026/reg/all
defense/passing: 32 rows, 15 columns

Opening defense - rushing: https://www.nfl.com/stats/team-stats/defense/rus

***
The cell below reads the stored .pkl files into pandas dataframes, where they can then be viewed or modified.
***

In [7]:
import pandas as pd

offensive_df = pd.read_pickle(
    "nfl_data/all_offensive_stats.pkl"
)

defensive_df = pd.read_pickle(
    "nfl_data/all_defensive_stats.pkl"
)

special_df = pd.read_pickle(
    "nfl_data/all_special_stats.pkl"
)

# Conclusion
The biggest lesson was that scraping depends on what the website allows. Our original plan was Pro Football Reference, but its CAPTCHAs blocked our scraper, so we had to pivot to NFL.com. We also learned that there are many ways to collect web data, and the right one depends on the site. For future students, an idea would be testing a scraper on your sources before committing to it in a proposal. We wish we had known how to get past CAPTCHAs, since that would have let us keep our first choice and would have given us more variability in the data we have access to.


***
Documentation Statement: We used ChatGPT's GPT-6 in accordance with AW's GenAI policy to write our scraping code and polish up our introduction and conclusion.